<a href="https://colab.research.google.com/github/Ramdharshan2007/DAA-Lab-Experiment/blob/main/8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random
import itertools
import time

# --- 1. Helper Functions ---
def generate_random_cities(n, grid_size=100):
    """Generates n random 2D coordinates representing cities."""
    return [(random.randint(0, grid_size), random.randint(0, grid_size)) for _ in range(n)]

def calculate_distance_matrix(cities):
    """Calculates the Euclidean distance between all pairs of cities."""
    n = len(cities)
    matrix = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = cities[i][0] - cities[j][0]
                dy = cities[i][1] - cities[j][1]
                matrix[i][j] = math.sqrt(dx**2 + dy**2)
    return matrix

def get_path_cost(path, matrix):
    """Calculates the total cost of a given Hamiltonian cycle."""
    cost = 0.0
    for i in range(len(path) - 1):
        cost += matrix[path[i]][path[i+1]]
    return cost

# --- 2. Nearest-Neighbour Heuristic ---
def nearest_neighbor(matrix, start_city=0):
    """Greedy approach to find an initial valid TSP tour."""
    n = len(matrix)
    unvisited = set(range(n))
    unvisited.remove(start_city)

    path = [start_city]
    current = start_city

    while unvisited:
        nearest = min(unvisited, key=lambda city: matrix[current][city])
        path.append(nearest)
        unvisited.remove(nearest)
        current = nearest

    path.append(start_city) # Return to start
    return path, get_path_cost(path, matrix)

# --- 3. 2-Opt Improvement Heuristic ---
def two_opt(path, matrix):
    """
    Iteratively improves a path by reversing segments (removing crossing edges)
    until no further improvements can be made.
    """
    best_path = path[:]
    best_cost = get_path_cost(best_path, matrix)
    improved = True

    while improved:
        improved = False
        # Iterate over all valid pairs of edges to swap
        for i in range(1, len(best_path) - 2):
            for j in range(i + 1, len(best_path) - 1):
                # Reverse the segment between i and j
                new_path = best_path[:i] + best_path[i:j][::-1] + best_path[j:]
                new_cost = get_path_cost(new_path, matrix)

                # If the reversal leads to a shorter tour, keep it
                if new_cost < best_cost:
                    best_path = new_path
                    best_cost = new_cost
                    improved = True
                    # Break to restart the while loop with the new best path
                    break
            if improved:
                break

    return best_path, best_cost

# --- 4. Brute Force (Exact Optimal) ---
def brute_force(matrix, start_city=0):
    """Evaluates all (N-1)! permutations to find the absolute minimum cost."""
    n = len(matrix)
    cities_to_visit = list(range(n))
    cities_to_visit.remove(start_city)

    min_cost = float('inf')
    best_path = None

    for perm in itertools.permutations(cities_to_visit):
        current_path = [start_city] + list(perm) + [start_city]
        current_cost = get_path_cost(current_path, matrix)

        if current_cost < min_cost:
            min_cost = current_cost
            best_path = current_path

    return best_path, min_cost

# --- Main Execution ---
if __name__ == '__main__':
    NUM_CITIES = 10

    # Generate 10 random cities and calculate distances
    random.seed(42) # Seed for reproducibility
    cities = generate_random_cities(NUM_CITIES)
    dist_matrix = calculate_distance_matrix(cities)

    print(f"--- TSP 2-Opt Heuristic vs Optimal ({NUM_CITIES} Cities) ---")

    # 1. Run Nearest Neighbour (Initial Solution)
    start_nn = time.perf_counter()
    nn_path, nn_cost = nearest_neighbor(dist_matrix)
    time_nn = time.perf_counter() - start_nn

    # 2. Run 2-Opt (Improvement)
    start_2opt = time.perf_counter()
    opt2_path, opt2_cost = two_opt(nn_path, dist_matrix)
    time_2opt = time.perf_counter() - start_2opt

    # 3. Run Brute Force (Absolute Optimal)
    print("Computing brute-force absolute optimal... (This evaluates 9! = 362,880 routes)")
    start_bf = time.perf_counter()
    bf_path, bf_cost = brute_force(dist_matrix)
    time_bf = time.perf_counter() - start_bf

    # 4. Display Results
    print("\n[1] Nearest Neighbour (Initial Greedy)")
    print(f"    Path: {' -> '.join(map(str, nn_path))}")
    print(f"    Cost: {nn_cost:.2f} | Time: {time_nn:.6f} s")

    print("\n[2] 2-Opt (Improvement Heuristic)")
    print(f"    Path: {' -> '.join(map(str, opt2_path))}")
    print(f"    Cost: {opt2_cost:.2f} | Time: {time_2opt:.6f} s")

    print("\n[3] Brute Force (Exact Optimal)")
    print(f"    Path: {' -> '.join(map(str, bf_path))}")
    print(f"    Cost: {bf_cost:.2f} | Time: {time_bf:.6f} s")

    # 5. Analysis
    print("\n--- Accuracy Inference ---")
    nn_error = ((nn_cost - bf_cost) / bf_cost) * 100
    opt2_error = ((opt2_cost - bf_cost) / bf_cost) * 100

    print(f"Initial NN Error:  +{nn_error:.2f}% worse than optimal")
    print(f"Improved 2-Opt Error: +{opt2_error:.2f}% worse than optimal")
    if opt2_cost == bf_cost:
        print("Success: The 2-Opt heuristic successfully converged to the absolute optimal solution!")

--- TSP 2-Opt Heuristic vs Optimal (10 Cities) ---
Computing brute-force absolute optimal... (This evaluates 9! = 362,880 routes)

[1] Nearest Neighbour (Initial Greedy)
    Path: 0 -> 6 -> 4 -> 7 -> 5 -> 2 -> 3 -> 9 -> 8 -> 1 -> 0
    Cost: 468.29 | Time: 0.000035 s

[2] 2-Opt (Improvement Heuristic)
    Path: 0 -> 4 -> 6 -> 7 -> 5 -> 2 -> 3 -> 8 -> 9 -> 1 -> 0
    Cost: 451.72 | Time: 0.000111 s

[3] Brute Force (Exact Optimal)
    Path: 0 -> 4 -> 7 -> 5 -> 1 -> 9 -> 8 -> 3 -> 2 -> 6 -> 0
    Cost: 370.44 | Time: 0.602772 s

--- Accuracy Inference ---
Initial NN Error:  +26.41% worse than optimal
Improved 2-Opt Error: +21.94% worse than optimal
